In [ ]:
import pandas as pd
import numpy as np
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from wordcloud import WordCloud


d:\Documentos\TRABAJOS_YO\UAEH\Proyecto Doctora Rosa\INTERFAZ_v1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Documentos\TRABAJOS_YO\UAEH\Proyecto Doctora Rosa\INTERFAZ_v1\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Denhy\.cache\huggingface\hub\datasets--PlanTL-GOB-ES--WikiCAT_esv2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate D

RuntimeError: Dataset scripts are no longer supported, but found WikiCAT_esv2.py

In [ ]:

nlp = spacy.load("es_core_news_md")

# 1. FUNCIÓN DE LIMPIEZA DE TEXTO CON SPACY
def limpiar_texto_spacy(texto):
    """
    Limpia texto usando spaCy: remove stopwords, puntuación, lematiza
    y filtra por POS tags relevantes.
    """
    if not isinstance(texto, str) or len(texto.strip()) == 0:
        return ""
    
    doc = nlp(texto.lower())
    palabras_limpias = []
    
    for token in doc:
        # Filtrar por POS tags (sustantivos, adjetivos, verbos)
        if (token.pos_ in ['NOUN', 'ADJ', 'VERB'] and 
            not token.is_stop and 
            not token.is_punct and 
            len(token.lemma_) > 2):
            palabras_limpias.append(token.lemma_)
    
    return " ".join(palabras_limpias)

# 2. FUNCIÓN TF-IDF + KMEANS
def aplicar_tfidf_kmeans(textos, n_clusters=5):
    """
    Aplica TF-IDF y clustering con KMeans
    """
    # Vectorización TF-IDF
    tfidf_vectorizer = TfidfVectorizer(
        max_features=1000,
        min_df=2,
        max_df=0.85,
        stop_words=list(nlp.Defaults.stop_words)
    )
    
    tfidf_matrix = tfidf_vectorizer.fit_transform(textos)
    
    # Clustering con KMeans
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )
    
    clusters_kmeans = kmeans.fit_predict(tfidf_matrix)
    
    # Métricas de evaluación
    silueta = silhouette_score(tfidf_matrix, clusters_kmeans)
    
    return {
        'vectorizer': tfidf_vectorizer,
        'matrix': tfidf_matrix,
        'clusters': clusters_kmeans,
        'silhouette_score': silueta,
        'feature_names': tfidf_vectorizer.get_feature_names_out()
    }

# 3. FUNCIÓN BERTopic CON SENTENCE TRANSFORMERS
def aplicar_bertopic(textos, use_hdbscan=True, n_clusters=5):
    """
    Aplica BERTopic con Sentence Transformers y HDBSCAN o KMeans
    """
    # Modelo de embeddings
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # Reducción de dimensionalidad
    umap_model = UMAP(
        n_components=5,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42
    )
    
    # Algoritmo de clustering
    if use_hdbscan:
        cluster_model = HDBSCAN(
            min_cluster_size=5,
            min_samples=2,
            cluster_selection_epsilon=0.1
        )
    else:
        from sklearn.cluster import KMeans
        cluster_model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    
    # Modelo BERTopic
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        verbose=True
    )
    
    # Entrenamiento
    topics, probabilities = topic_model.fit_transform(textos)
    
    return {
        'model': topic_model,
        'topics': topics,
        'probabilities': probabilities,
        'info': topic_model.get_topic_info()
    }


def visualizar_resultados(textos, clusters, topic_model=None):
 
    # Word Cloud por cluster
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Word Cloud general
    all_text = ' '.join(textos)
    wordcloud = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        stopwords=list(nlp.Defaults.stop_words)
    ).generate(all_text)
    
    axes[0, 0].imshow(wordcloud, interpolation='bilinear')
    axes[0, 0].set_title('Word Cloud - Todos los documentos')
    axes[0, 0].axis('off')
    
    # Distribución de clusters
    unique, counts = np.unique(clusters, return_counts=True)
    axes[0, 1].bar(unique, counts)
    axes[0, 1].set_title('Distribución de Clusters')
    axes[0, 1].set_xlabel('Cluster')
    axes[0, 1].set_ylabel('Número de documentos')

    if topic_model is not None:
        try:
            # Visualización de tópicos
            topic_vis = topic_model.visualize_topics()
            topic_vis.show()
    
            doc_vis = topic_model.visualize_documents(textos)
            doc_vis.show()
        except:
            print("No se pudieron generar visualizaciones interactivas")
    
    plt.tight_layout()
    plt.show()


def ejecutar_pipeline_topic_modeling(datos, columna_texto, use_hdbscan=True):
  
    print("Limpieza de texto")
    textos = datos[columna_texto].tolist()
    textos_limpios = [limpiar_texto_spacy(texto) for texto in textos]
    
    print(f"Textos originales: {len(textos)}")
    print(f"Textos después de limpieza: {len([t for t in textos_limpios if t.strip()])}")
    
    # Filtrar textos vacíos
    textos_filtrados = [t for t in textos_limpios if len(t.strip()) > 10]
    indices_validos = [i for i, t in enumerate(textos_limpios) if len(t.strip()) > 10]
    
    print("Tf idf")
    resultados_tfidf = aplicar_tfidf_kmeans(textos_filtrados)
    print(f"Silhouette Score (KMeans): {resultados_tfidf['silhouette_score']:.3f}")
    
    print("Bertopic")
    resultados_bertopic = aplicar_bertopic(textos_filtrados, use_hdbscan=use_hdbscan)
    
    print("=== VISUALIZACIÓN ===")
    visualizar_resultados(textos_filtrados, resultados_tfidf['clusters'], resultados_bertopic['model'])
    
    # Crear DataFrame con resultados
    df_resultados = pd.DataFrame({
        'texto_original': [textos[i] for i in indices_validos],
        'texto_limpio': textos_filtrados,
        'cluster_tfidf_kmeans': resultados_tfidf['clusters'],
        'cluster_bertopic': resultados_bertopic['topics']
    })
    
    return {
        'df_resultados': df_resultados,
        'resultados_tfidf': resultados_tfidf,
        'resultados_bertopic': resultados_bertopic
    }


def mostrar_topicos_bertopic(resultados_bertopic, n_palabras=10):
 
    topic_info = resultados_bertopic['info']
    print("topicos")
    
    for index, row in topic_info.iterrows():
        if row['Topic'] != -1:  # Excluir outliers
            print(f"\nTópico {row['Topic']} - {row['Count']} documentos")
            topic_words = resultados_bertopic['model'].get_topic(row['Topic'])
            for palabra, score in topic_words[:n_palabras]:
                print(f"  {palabra}: {score:.3f}")


    # Ejecutar pipeline
    resultados = ejecutar_pipeline_topic_modeling(
        datos_ejemplo, 
        'texto', 
        use_hdbscan=True
    )
    
    # Mostrar tópicos
    mostrar_topicos_bertopic(resultados['resultados_bertopic'])
    
    # Guardar resultados
    resultados['df_resultados'].to_csv('resultados_topic_modeling.csv', index=False)
    print("\nResultados guardados en 'resultados_topic_modeling.csv'")